TMDB Assignment

Task 1 — Build the Pipeline

In [13]:
import os, json, time, sqlite3, requests
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import requests
from google.colab import userdata

# Get API key from Colab secrets
API_KEY = userdata.get('TMDB_key')  # here my key is stored in the googles secret manager

# API_KEY = 'Enter your api key here'
BASE_URL = "https://api.themoviedb.org/3"

In [11]:
# Fetched the data using the correct endpoint by using the suitable parameters.

all_movies = {}

for page in range(1, 3):
    response = requests.get(
        f"{BASE_URL}/discover/movie",
        params={
            "api_key": API_KEY,
            "sort_by": "popularity.desc",
            "page": page
        }
    )

    data = response.json().get("results", [])

    # here we added the key value pairs in the empty dictonary
    for movie in data:
        all_movies[movie["id"]] = movie



In [16]:
# Now we will create a dataframe
import pandas as pd
movies_list = []

for movie in all_movies.values():
    movies_list.append({
        "movie_id": movie.get("id"),
        "title": movie.get("title"),
        "original_language": movie.get("original_language"),
        "release_date": movie.get("release_date"),
        "popularity": movie.get("popularity"),
        "vote_average": movie.get("vote_average"),
        "vote_count": movie.get("vote_count"),
        "genre": movie.get("genre_ids")
    })

movies_df = pd.DataFrame(movies_list)

movies_df.head()

,movie_id,title,original_language,release_date,popularity,vote_average,vote_count,genre
0,1523145,Your Heart Will Be Broken,ru,2026-03-26,1165.1266,6.900,24,"[10749, 18]"
1,83533,Avatar: Fire and Ash,en,2025-12-17,503.6662,7.300,2074,"[878, 12, 14]"
2,1115544,Mike & Nick & Nick & Alice,en,2026-03-14,352.9397,6.700,109,"[35, 878, 80]"
3,1084187,Pretty Lethal,en,2026-03-13,311.0726,6.745,157,"[28, 53, 10402]"
4,1297842,GOAT,en,2026-02-11,297.8194,7.700,249,"[16, 35, 10751]"


In [17]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           40 non-null     int64  
 1   title              40 non-null     object 
 2   original_language  40 non-null     object 
 3   release_date       40 non-null     object 
 4   popularity         40 non-null     float64
 5   vote_average       40 non-null     float64
 6   vote_count         40 non-null     int64  
 7   genre              40 non-null     object 
dtypes: float64(2), int64(2), object(4)
memory usage: 2.6+ KB


In [18]:
movies_df.isnull().sum()

,0
movie_id,0
title,0
original_language,0
release_date,0
popularity,0
vote_average,0
vote_count,0
genre,0


In [22]:
conn = sqlite3.connect("movies.db")

# Convert list to string
movies_df["genre"] = movies_df["genre"].astype(str)

movies_df.to_sql("movies", conn, if_exists="replace", index=False)

conn.close()

print("Data saved successfully!")

Data saved successfully!


Task 2 — Perform EDA

In [24]:
# Loading the movies table into a dataframe
conn = sqlite3.connect("movies.db")
movies_df = pd.read_sql("SELECT * FROM movies", conn)
conn.close()

movies_df.head(5)

,movie_id,title,original_language,release_date,popularity,vote_average,vote_count,genre
0,1523145,Your Heart Will Be Broken,ru,2026-03-26,1165.1266,6.900,24,"[10749, 18]"
1,83533,Avatar: Fire and Ash,en,2025-12-17,503.6662,7.300,2074,"[878, 12, 14]"
2,1115544,Mike & Nick & Nick & Alice,en,2026-03-14,352.9397,6.700,109,"[35, 878, 80]"
3,1084187,Pretty Lethal,en,2026-03-13,311.0726,6.745,157,"[28, 53, 10402]"
4,1297842,GOAT,en,2026-02-11,297.8194,7.700,249,"[16, 35, 10751]"


In [25]:
movies_df.describe()

,movie_id,popularity,vote_average,vote_count
count,4.000000e+01,40.000000,40.000000,40.000000
mean,1.113495e+06,171.956862,6.747325,1018.875000
std,3.429845e+05,188.115375,0.984482,3406.274807
min,8.353300e+04,54.824200,4.149000,1.000000
25%,8.725558e+05,84.560625,6.214250,38.750000
50%,1.169668e+06,100.782100,6.952000,277.500000
75%,1.319024e+06,180.426900,7.305250,791.750000
max,1.634301e+06,1165.126600,8.500000,21722.000000


In [27]:
# Count the number of movies per genre
genre_counts = (
    movies_df["genre"]
    .str.split(",")
    .explode()
    .value_counts()
)

genre_counts

,count
genre,
53],11
[28,9
35,6
18],5
[27,5
80,5
[878,5
[16,4
18,4


In [28]:
# Identify columns with missing values
missing_values = movies_df.isnull().sum()

missing_values

,0
movie_id,0
title,0
original_language,0
release_date,0
popularity,0
vote_average,0
vote_count,0
genre,0
